# Akhilesh
## DATA 266 - Homework 1 CUDA

Personal parameters: `SID4=6812`, `SEED=6812`, `SLICE=812`, `HP_ID=2`, `CLS_A=2`, `CLS_B=9`.

This notebook builds and runs `matmul_cuda.cu`. The kernel uses a 2D grid of blocks and a 2D block of threads. Each thread computes one output element `C[row, col]` by taking the dot product of one row of `A` and one column of `B`.


In [4]:
import os
import subprocess
from pathlib import Path

print("Working directory:", Path.cwd())
print("CUDA source exists:", Path("matmul_cuda.cu").exists())


Working directory: /content
CUDA source exists: True


In [5]:
def run(cmd):
    print("$", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed.returncode

run("nvidia-smi")
run("nvcc --version")


$ nvidia-smi
Tue Sep  1 06:25:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------------

0

## Build and run

Run these cells in Google Colab or another environment where `nvcc` is installed. On an A100, `-arch=sm_80` matches the GPU generation. The program prints CPU time, GPU kernel time, transfer time, end-to-end speedup, and a small correctness check for the sizes where the CPU result is computed.


In [6]:
run("nvcc -O3 -arch=sm_80 matmul_cuda.cu -o matmul_cuda")
run("./matmul_cuda")


$ nvcc -O3 -arch=sm_80 matmul_cuda.cu -o matmul_cuda
$ ./matmul_cuda
Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),Speedup,Check
256,21.643,0.121,0.279,54.020,max error 0.000015
1024,3305.822,0.903,3.013,844.163,max error 0.000092
4096,657903.697,67.836,42.927,5939.709,max error 0.001221



0

## Profiling

Use one profiler that is available. In Colab, `nvprof` is often unavailable on newer GPUs, so `nsys` or `ncu` may also be missing. Record the output from whichever command works.


In [7]:
profile_commands = [
    "nsys profile --stats=true ./matmul_cuda",
    "ncu ./matmul_cuda",
    "nvprof ./matmul_cuda",
]

for cmd in profile_commands:
    rc = run(cmd)
    if rc == 0:
        print("Profiler used:", cmd.split()[0])
        break
else:
    print("No CUDA profiler command succeeded in this environment.")


$ nsys profile --stats=true ./matmul_cuda
/bin/sh: 1: nsys: not found

$ ncu ./matmul_cuda
Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),Speedup,Check
==PROF== Connected to process 5923 (/content/matmul_cuda)
==PROF== Profiling "matmul_kernel" - 0: 0%....50%....100% - 10 passes
256,21.643,1566.781,0.321,0.014,max error 0.000015
==PROF== Profiling "matmul_kernel" - 1: 0%....50%....100% - 10 passes
1024,3351.161,679.720,3.022,4.908,max error 0.000092
==PROF== Profiling "matmul_kernel" - 2: 0%....50%....100% - 10 passes
4096,756880.099,1362.228,43.106,538.577,max error 0.001221
==PROF== Disconnected from process 5923
[5923] matmul_cuda@127.0.0.1
  matmul_kernel(const float *, const float *, float *, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency  